## Lab 4: Gaps and merge

Today we looked for missing values, decided what to do with them, and put two tables together with merge. 
The most important lesson was to ask why a value is missing, and to check the numbers after every step. 
In this lab you try it all on new data: the loans and the members of a library. 
The library and its data are made up, but the problems are of the kinds that real data has. 
This lab has two parts, and you choose.

Part 1: Practise repeats the steps from today on the new tables. The problems are of the same kinds as today, 
in new places. Do this part if you want to try the whole routine once more on your own.

Part 2: Go deeper is for you who felt "I understood this, and I want to go further". 
The questions are more open, and you have to decide what is right when the data does not give a clear answer.

I do not expect you to do both parts. It is your own choice, and you are welcome to contact me if
you want to talk about which part to take.
Download library_loans.csv and library_members.csv from Teams and save them in your data
folder. Work in a new notebook. Keep the originals: load each table into ..._raw and work on a
copy. Write a short comment above each step, in your own words. If you get stuck: read the error
message from the bottom up, print the thing you are working with, and look at the lecture
notebook. Then ask.

## Part 1: Practise
Look for gaps, decide what to do with them, and merge the two tables, with the same steps as
today.

## Tools you need

loans = pd.read_csv('data/library_loans.csv', parse_dates=['loan_date', 'return_date'])
members_raw = pd.read_csv('data/library_members.csv') members = members_raw.copy()
table.isna().sum() (table.isna().mean() * 100).round(1) table['x'].notna()
table['x'].where(table['x'].between(1920, 2015)) # outside the range -> NaN
table.dropna() table.dropna(subset=['x']) table['x'].fillna('Unknown')
table['x'].median() table['x'].std()
loans.merge(members, on='member_id', how='left', validate='many_to_one', indicator=True)
merged['_merge'].value_counts() # both / left_only / right_only
table.drop_duplicates().reset_index(drop=True) table.groupby('x')['y'].count()

## Tasks
1 Load the loans and look for gaps

Load library_loans.csv with parse_dates for both date columns. Print the shape and the dtypes. 
Then count the missing values in every column, and show them as percent. 
Which two columns have gaps, and how many?

In [3]:
import pandas as pd

# Load the loans dataset with parse_dates for both date columns
loans_raw = pd.read_csv('data/library_loans_clean.csv', parse_dates=['loan_date', 'return_date'])
loans = loans_raw.copy()

# Print the shape and dtypes
print("   --- Shape of library_loans ---")
print(loans.shape)

print("\n--- Data types --- ")
print(loans.dtypes)

# Count the missing values in every column
missing_counts = loans.isna().sum()

# Show missing values as percentages (rounded to 1 dec. place)
missing_percentages = (loans.isna().mean() * 100).round(1)

# Combine into a clean summary table for inspection
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage (%)': missing_percentages
})

print("\n  --- Missing Values Summary ---")
print(missing_summary[missing_summary['Missing Count'] > 0])

   --- Shape of library_loans ---
(600, 8)

--- Data types --- 
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object

  --- Missing Values Summary ---
             Missing Count  Missing Percentage (%)
loan_date                3                     0.5
return_date             36                     6.0


## 2 Two kinds of missing

Show the loans where loan_date is missing. 
Then look at return_date: how many are missing and the book is back (returned is True), and how many are missing and the book is not back? 
Which of the two kinds is "missing for a reason", and which is "missing by accident"? 
Create a column has_return_date with notna() and count it.

In [4]:
# 2. Two kinds of missing

# Show loans where loan_date is missing
missing_loans = loans[loans['loan_date'].isna()]
print(f"Number of rows with missing loan_date: {len(missing_loans)}")
print(missing_loans.head())

# Analyze missing return_date based on whether the book is returned (True/False)
# How many are missing return_date and the book is back (returned == True)?
missing_returned_true = loans[loans['return_date'].isna() & (loans['returned'] == True)]
print(f"\n Missing return_date while returned == True: {len(missing_returned_true)}")

# How many are missing return_date and the book IS NOT back (returned == False)?
missing_returned_false = loans[loans['return_date'].isna() & (loans['returned'] == False)]
print(f"Missing return_date while returned == False: {len(missing_returned_false)}")

# Create a column 'has_return_date' with notna() and count it
loans['has_return_date'] = loans['return_date'].notna()

print("\n--- Count of has_return_date ---")
print(loans['has_return_date'].value_counts())

Number of rows with missing loan_date: 3
     loan_id member_id               title     genre loan_date return_date  \
462  L-30463     M0007  Blue Moon Protocol    Sci-fi       NaT  2026-05-24   
467  L-30468     M0018  Blue Moon Protocol    Sci-fi       NaT  2026-06-08   
571  L-30572     M0090  Mimmi and the Moon  Children       NaT  2026-06-26   

     late_fee  returned  
462       0.0      True  
467       0.0      True  
571       0.0      True  

 Missing return_date while returned == True: 6
Missing return_date while returned == False: 30

--- Count of has_return_date ---
has_return_date
True     564
False     36
Name: count, dtype: int64


## 3 Look at the members

Load library_members.csv, make a copy, and print the shape. 
Run info() and describe() on birth_year. 
Something is wrong with the minimum and the maximum. 
Count how many members have each strange value, 
and check for duplicates in the whole row and in member_id. 
Write the list of problems as a comment.

In [ ]:
import pandas as pd

# Load library_members.csv, make a copy, and inspect
members_raw = pd.read_csv('data/library_members.csv')
members = members_raw.copy()

# Print shape, info, and describe birth_year
print("--- Shape of library_members ---")
print(members.shape)

print("\n--- Members Info ---")
members.info()

print("\n--- Describe birth_year ---")
print(members['birth_year'].describe())

# Count strange values in birth_year
print("\n--- Value counts of birth_year ---")
print(members['birth_year'].value_counts(dropna=False))

# Check for duplicates in the whole row and in member_id
print("\n--- Duplicate Check ---")
print("Full row duplicates:", members.duplicated().sum())
print("Duplicates by member_id:", members.duplicated(subset='member_id').sum())

# --- LIST OF PROBLEMS (Comment) ---
# 1. Unrealistic birth years: The minimum (1900) and maximum values (2030) in 'birth_year' contain impossible years.
# 2. Full row duplicates (4): There are completely identical rows repeated in the dataset.
# 3. ID duplicates (4): There are duplicate rows sharing the exact same 'member_id', 


--- Shape of library_members ---
(148, 3)

--- Members Info ---
<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB

--- Describe birth_year ---
count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64

--- Value counts of birth_year ---
birth_year
NaN       18
1900.0     8
1990.0     7
1979.0     7
1988.0     6
1987.0     4
1994.0     4
1975.0     4
1992.0     4
2010.0     4
1999.0     4
1984.0     4
1986.0     3
2001.0     3
1991.0     3
1995.0     3
1968.0     3
1997.0     3
1976.0     3
2003.0     3
1982.0     3
2005.0     3
1985.0     3
1989.0     3
2

## 4 Placeholders and impossible values
Nobody in this library is born in 1900, and nobody is born in the future. 
Use where and between(1920, 2015) to turn those values into missing values. 
How many birth years are missing now? 
Run describe() again and compare it with before.

In [6]:
# 4. Placeholders and impossible values

# Count missing birth years BEFORE cleaning
missing_before = members['birth_year'].isna().sum()
print(f"Missing birth years before cleaning: {missing_before}")

# Use .where() and .between() to turn out-of-range years into NaN

members['birth_year'] = members['birth_year'].where(
    members['birth_year'].between(1920, 2015)
)

# Count missing birth years AFTER cleaning
missing_after = members['birth_year'].isna().sum()
print(f"Missing birth years after cleaning: {missing_after}")
print(f"Newly converted missing values: {missing_after - missing_before}")

# Run describe() again to compare with before
print("\n--- Describe birth_year after cleaning ---")
print(members['birth_year'].describe())

Missing birth years before cleaning: 18
Missing birth years after cleaning: 29
Newly converted missing values: 11

--- Describe birth_year after cleaning ---
count     119.000000
mean     1987.815126
std        12.062666
min      1957.000000
25%      1979.000000
50%      1988.000000
75%      1996.500000
max      2010.000000
Name: birth_year, dtype: float64


## 5 Remove, fill, or leave

(a) Without saving the result, see how many rows are left with dropna(), 
with dropna(subset=['branch']) and with dropna(subset=['birth_year']).

(b) Fill the missing branch with 'Unknown' and count the branches.
(c) Try filling the missing birth_year with the median, on the side, in a separate variable.

Compare the mean and the standard deviation before and after. What happens to the
spread, and why? What do you decide to do with birth_year?

In [7]:
# 5. Remove, fill, or leave

# (a) Testing different dropna() scenarios (without saving) ---
print(f"Original shape of members table: {members.shape}")
print(f"Rows left with dropna() (drops any row with ANY missing value): {members.dropna().shape[0]}")
print(f"Rows left with dropna(subset=['branch']): {members.dropna(subset=['branch']).shape[0]}")
print(f"Rows left with dropna(subset=['birth_year']): {members.dropna(subset=['birth_year']).shape[0]}")

# (b) Fill missing branch values with 'Unknown' and count ---
members['branch'] = members['branch'].fillna('Unknown')

print("\n--- Branch counts after filling NaNs with 'Unknown' ---")
print(members['branch'].value_counts(dropna=False))

# (c) Filling missing birth_year with the median in a separate variable 
# and calculate statistics BEFORE filling
mean_before = members['birth_year'].mean()
std_before = members['birth_year'].std()

# Fill missing values with the median in a side variable
birth_year_median_filled = members['birth_year'].fillna(members['birth_year'].median())

# Calculate statistics AFTER filling
mean_after = birth_year_median_filled.mean()
std_after = birth_year_median_filled.std()

print("\n--- Comparing Birth Year Statistics ---\n")
print(f"Before filling -> Mean: {mean_before:.2f}, Std: {std_before:.2f}")
print(f"After filling  -> Mean: {mean_after:.2f}, Std: {std_after:.2f}")

Original shape of members table: (148, 3)
Rows left with dropna() (drops any row with ANY missing value): 107
Rows left with dropna(subset=['branch']): 136
Rows left with dropna(subset=['birth_year']): 119

--- Branch counts after filling NaNs with 'Unknown' ---
branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64

--- Comparing Birth Year Statistics ---

Before filling -> Mean: 1987.82, Std: 12.06
After filling  -> Mean: 1987.85, Std: 10.81


# 6 Merge, and check

Print the shape of both tables.
Merge the loans with the members with how='left' and without validate. 
How many rows do you get, and why is it not 600? 
Calculate the total of late_fee before and after the merge. 
Then merge again with validate='many_to_one' and read the error from the bottom up.
On purpose: The merge with validate should give a MergeError.

In [8]:
# 6. Merge, and check

# Print the shape of both tables
print("Loans table shape:", loans.shape)
print("Members table shape:", members.shape)

# Merge loans with members (left join) WITHOUT validate
merged_naive = loans.merge(members, on='member_id', how='left')
print("Shape of merged table (without validate):", merged_naive.shape)

# Calculate total late_fee before and after the merge
total_fee_before = loans['late_fee'].sum()
total_fee_after = merged_naive['late_fee'].sum()

print(f"\nTotal late fee before merge: {total_fee_before:.2f}")
print(f"Total late fee after merge:  {total_fee_after:.2f}")

# Try merging with validate='many_to_one' on purpose to see the MergeError
try:
    loans.merge(members, on='member_id', how='left', validate='many_to_one')
except Exception as e:
    print("\n--- Caught expected MergeError (read from bottom up) ---")
    print(e)

Loans table shape: (600, 9)
Members table shape: (148, 3)
Shape of merged table (without validate): (616, 11)

Total late fee before merge: 7487.50
Total late fee after merge:  7725.00

--- Caught expected MergeError (read from bottom up) ---
Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 member_id
    M0015
    M0033
    M0065
    M0071 ...


## 7 Fix the key, and find who is missing

Remove the duplicates from members and merge again with validate='many_to_one' and indicator=True. 
Check the shape and the total of late_fee. 
How many loans have left_only, how many different members is that, and how many kr in late fees? 
What would an inner merge have given for the shape and the total?

In [10]:
# 7. Fix the key, and find who is missing

# Remove duplicates from the members table
members = members.drop_duplicates().drop_duplicates(subset='member_id', keep='first').reset_index(drop=True)

# Merge loans with cleaned members using left join, validate, and indicator
merged = loans.merge(
    members, 
    on='member_id', 
    how='left', 
    validate='many_to_one', 
    indicator=True
)

# Check shape and total late_fee
print(f"Shape of merged table: {merged.shape}")
print(f"Total late fee after merge: {merged['late_fee'].sum():.2f} kr")

# Analyze the _merge indicator column
print("\n--- Merge Indicator Breakdown ---")
print(merged['_merge'].value_counts())

# Extract rows where merge status is 'left_only' (loans whose member_id wasn't found in members)
left_only_loans = merged[merged['_merge'] == 'left_only']
num_left_only_loans = len(left_only_loans)
num_unique_members_missing = left_only_loans['member_id'].nunique()
total_fee_left_only = left_only_loans['late_fee'].sum()

print(f"\n--- 'left_only' Analysis ---")
print(f"Number of loans with 'left_only': {num_left_only_loans}")
print(f"Number of different members missing from the members table: {num_unique_members_missing}")
print(f"Total late fees from these unlinked loans: {total_fee_left_only:.2f} kr")

# What would an inner merge have given?
inner_merged = loans.merge(
    members, 
    on='member_id', 
    how='inner', 
    validate='many_to_one'
)
print(f"\n--- Inner Merge Projection ---")
print(f"Inner merge shape: {inner_merged.shape}")
print(f"Inner merge total late fee: {inner_merged['late_fee'].sum():.2f} kr")

Shape of merged table: (600, 12)
Total late fee after merge: 7487.50 kr

--- Merge Indicator Breakdown ---
_merge
both          569
left_only      31
right_only      0
Name: count, dtype: int64

--- 'left_only' Analysis ---
Number of loans with 'left_only': 31
Number of different members missing from the members table: 6
Total late fees from these unlinked loans: 347.50 kr

--- Inner Merge Projection ---
Inner merge shape: (569, 11)
Inner merge total late fee: 7140.00 kr


## 8 Use the merged table

Give the loans without a member record the branch 'No member record' (fillna). 
Count the loans per branch, biggest first. Which branch has the most loans? 
Write one sentence about the two groups Unknown and No member record: why are they not the same thing?

In [11]:
# 8. Use the merged table

# Fill missing branch values (for loans without a member record) with 'No member record'
merged['branch'] = merged['branch'].fillna('No member record')

# Count the loans per branch, sorted from biggest to smallest
branch_counts = merged['branch'].value_counts()

print("--- Loans per Branch ---")
print(branch_counts)

# Identify the branch with the most loans
top_branch = branch_counts.index[0]
top_branch_count = branch_counts.iloc[0]
print(f"\nBranch with the most loans: '{top_branch}' with {top_branch_count} loans.")

--- Loans per Branch ---
branch
Central             144
South               130
North               126
East                 72
Harbour              51
Unknown              46
No member record     31
Name: count, dtype: int64

Branch with the most loans: 'Central' with 144 loans.


## Expected

Task 1: (600, 8). loan_date has 3 missing (0.5 percent) and return_date has 36 (6.0 percent). The
other columns have none.

Task 2: 3 rows with a missing loan_date (L-30463, L-30468 and L-30572). 30 loans have no return
date and are not back (missing for a reason), and 6 have no return date but are back (missing by
accident). has_return_date: True 564, False 36.

Task 3: (148, 3). branch has 136 values and birth_year has 130. birth_year: min 1900, max 2030,
mean 1983.4. 8 members have 1900 and 3 have 2030. 4 duplicate rows and 4 duplicate member_id.

Task 4: 29 missing (18 + 8 + 3). describe(): count 119, mean 1987.82, min 1957, max 2010.

Task 5: (a) (107, 3), (136, 3) and (119, 3). (b) Central 40, South 34, North 32, East 17, Harbour 13,
Unknown 12. (c) Median 1988.0. The mean is 1987.82 before and 1987.85 after, the standard
deviation goes from 12.06 to 10.81.

Task 6: (600, 9) (the 8 original columns plus has_return_date) and (148, 3). The merge gives (616,
11): 16 extra rows, because 4 members are in the table twice. The total of late_fee goes from
7487.5 to 7725.0 (237.5 too much). The validate merge: MergeError: Merge keys are not unique
in right dataset; not a many-to-one merge.

Task 7: after the fix (144, 3), and the merge gives (600, 12) with indicator (600, 11 without). Total
7487.5. left_only 31 loans, from 6 different members, 347.5 kr. With inner: (569, 11) and 7140.0.

Task 8: Central 144, South 130, North 126, East 72, Harbour 51, Unknown 46, No member record
31. Unknown is a member who exists but has no branch. No member record is a loan whose member is
not in the table at all.

## Part 2: Go deeper

Here you ask what the gaps mean, and you document your decisions. There are two new tools
(dt.year and how='outer'), and they are in the box below.

Tools you need

merged['loan_date'].dt.year - merged['birth_year'] # age at the loan
loans.merge(members, on='member_id', how='outer', indicator=True) # keep rows from BOTH tables
members['member_id'].isin(loans['member_id']) # is the member in the loans?
table.groupby('x')['y'].agg(['count', 'mean'])

## Tasks ## 

## 1 Who is missing, and does it matter?

Look at the loans whose member is not in the member table. 
How many is that, what share of all loans, and what share of all late fees? 
Which genres do they borrow? 
Does it look like a pattern, or like chance? 
Write in a comment who you would ask, and what you would ask, to solve it.

In [13]:
# Part 2 - Task 1: Who is missing, and does it matter?

# Perform a left merge with indicator=True to isolate unlinked loans
merged_analysis = loans.merge(
    members.drop_duplicates(subset='member_id'), 
    on='member_id', 
    how='left', 
    validate='many_to_one', 
    indicator=True
)

# Isolate loans where the member is not in the member table ('left_only')
missing_members_loans = merged_analysis[merged_analysis['_merge'] == 'left_only']

# Calculate share of loans and share of late fees
total_loans_count = len(merged_analysis)
missing_loans_count = len(missing_members_loans)
missing_loans_share = (missing_loans_count / total_loans_count) * 100

total_fees = merged_analysis['late_fee'].sum()
missing_fees = missing_members_loans['late_fee'].sum()
missing_fees_share = (missing_fees / total_fees) * 100 if total_fees > 0 else 0

print(f"Number of loans with missing members: {missing_loans_count} ({missing_loans_share:.2f}% of all loans)")
print(f"Late fees from missing members: {missing_fees:.2f} kr ({missing_fees_share:.2f}% of total late fees)")

# Analyze which genres these unlinked loans borrow
print("\n--- Genre breakdown for missing-member loans ---")
print(missing_members_loans['genre'].value_counts(dropna=False))

# --- Write in a comment who you would ask, and what you would ask, to solve it ---
# Who to ask: The library's IT administrator or front-desk manager.
# What to ask: "Are these unlinked loan records created by guest/temporary checkouts 
# where a full member profile was never registered? ...


Number of loans with missing members: 31 (5.17% of all loans)
Late fees from missing members: 347.50 kr (4.64% of total late fees)

--- Genre breakdown for missing-member loans ---
genre
Children    10
Sci-fi       5
Crime        4
Fantasy      4
Romance      4
History      4
Name: count, dtype: int64


## 2.2 Members without loans

Some members may never have borrowed a book. 
Find out with an outer merge and indicator=True: how many are right_only? 
Then check the same thing in another way, with isin(). 
Which of the two ways do you prefer, and why?

In [12]:
# Part 2 - Task 2: Members without loans

# Using an outer merge with indicator=True
members_clean = members.drop_duplicates(subset='member_id').copy()

outer_merged = loans.merge(
    members_clean, 
    on='member_id', 
    how='outer', 
    indicator=True
)

# 'right_only' means the row exists in the members table, but had no match in the loans table
right_only_count = (outer_merged['_merge'] == 'right_only').sum()
print(f"Number of members without loans (Outer Merge - 'right_only'): {right_only_count}")


# Using the .isin() method
# Check which member IDs from the members table are NOT present in the loans table...

members_without_loans = members_clean[~members_clean['member_id'].isin(loans['member_id'])]
isin_count = len(members_without_loans)

print(f"Number of members without loans (isin() method): {isin_count}")

Number of members without loans (Outer Merge - 'right_only'): 1
Number of members without loans (isin() method): 1
